In [ ]:
import ast
import threading
from flask import Flask, request, render_template_string
import pandas as pd
import numpy as np
from neo4j import GraphDatabase
from neo4j.exceptions import ServiceUnavailable
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from transformers import pipeline
import warnings

# Suppress warnings
warnings.filterwarnings("ignore")

app = Flask(__name__)

### CONFIGURATION ###
NEO4J_URI = "bolt://localhost:7687"
NEO4J_USER = "neo4j"
NEO4J_PASSWORD = "argentic"  # Update with your actual password

# CSV file paths (all in the adjusted_datasets folder)
CITIES_CSV = "adjusted_datasets/adjusted_cities.csv"
FLIGHTS_CSV = "adjusted_datasets/adjusted_flights.csv"
HOTELS_CSV = "adjusted_datasets/adjusted_hotels.csv"
RESTAURANTS_CSV = "adjusted_datasets/adjusted_restaurants.csv"
PREFERENCES_CSV = "adjusted_datasets/preferences.csv"
USERS_CSV = "adjusted_datasets/users.csv"
PASSPORTS_CSV = "adjusted_datasets/adjusted_passports.csv"
HISTORIES_CSV = "adjusted_datasets/histories.csv"

# Global conversation history for the chat interface
conversation_history = []

# ----------------------------
# 1. Connect to Neo4j
# ----------------------------
try:
    driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
    with driver.session() as session:
        session.run("RETURN 1")
    print("Connected to Neo4j successfully.")
except ServiceUnavailable as e:
    print(f"Neo4j connection error: {e}")
    print("Please ensure Neo4j is running and accessible at", NEO4J_URI)
    print("You can start Neo4j by running 'neo4j start' in your command line")
    exit(1)

def get_nodes(label):
    """Retrieve all nodes of a given label from Neo4j."""
    with driver.session() as session:
        result = session.run(f"MATCH (n:{label}) RETURN n")
        return [record["n"] for record in result]

# ----------------------------
# 2. Node and Relationship Creation Functions
# ----------------------------
def create_city_node(tx, props):
    query = """
    MERGE (c:City {city_id: $city_id})
    SET c += $props
    """
    tx.run(query, city_id=props["city_id"], props=props)

def create_flight_node(tx, props):
    query = """
    MERGE (f:Flight {flight_id: $flight_id})
    SET f += $props
    """
    tx.run(query, flight_id=props["flight_id"], props=props)

def create_hotel_node(tx, props):
    query = """
    MERGE (h:Hotel {hotel_id: $hotel_id})
    SET h += $props
    """
    tx.run(query, hotel_id=props["hotel_id"], props=props)

def create_restaurant_node(tx, props):
    query = """
    MERGE (r:Restaurant {restaurant_id: $restaurant_id})
    SET r += $props
    """
    tx.run(query, restaurant_id=props["restaurant_id"], props=props)

def create_preference_node(tx, props):
    query = """
    MERGE (p:Preference {preference_id: $preference_id})
    SET p += $props
    """
    tx.run(query, preference_id=props["preference_id"], props=props)

def create_user_node(tx, props):
    user_id = props.get("User_ID") or props.get("user_id")
    if not user_id:
        raise KeyError("User_ID not found in props for user node")
    query = """
    MERGE (u:User {User_ID: $user_id})
    SET u += $props
    """
    tx.run(query, user_id=user_id, props=props)

def create_passport_node(tx, props):
    query = """
    MERGE (pp:Passport {passport_id: $passport_id})
    SET pp += $props
    """
    tx.run(query, passport_id=props["passport_id"], props=props)

def create_history_node(tx, props):
    query = """
    MERGE (h:History {history_id: $history_id})
    SET h += $props
    """
    tx.run(query, history_id=props["history_id"], props=props)

def create_relationship(tx, label_from, key_from, value_from, rel_type, label_to, key_to, value_to):
    query = f"""
    MATCH (a:{label_from} {{{key_from}: $value_from}})
    MATCH (b:{label_to} {{{key_to}: $value_to}})
    MERGE (a)-[r:{rel_type}]->(b)
    """
    tx.run(query, value_from=value_from, value_to=value_to)

# ----------------------------
# 3. Build Graph: Merge Nodes and Create Relationships
# ----------------------------
def build_graph():
    try:
        with driver.session() as session:
            # Load CSV files
            cities_df = pd.read_csv(CITIES_CSV)
            flights_df = pd.read_csv(FLIGHTS_CSV)
            hotels_df = pd.read_csv(HOTELS_CSV)
            restaurants_df = pd.read_csv(RESTAURANTS_CSV)
            preferences_df = pd.read_csv(PREFERENCES_CSV)
            users_df = pd.read_csv(USERS_CSV)
            passports_df = pd.read_csv(PASSPORTS_CSV)
            histories_df = pd.read_csv(HISTORIES_CSV)
            
            # Create nodes
            for _, row in cities_df.iterrows():
                session.execute_write(create_city_node, row.to_dict())
            for _, row in flights_df.iterrows():
                session.execute_write(create_flight_node, row.to_dict())
            for _, row in hotels_df.iterrows():
                session.execute_write(create_hotel_node, row.to_dict())
            for _, row in restaurants_df.iterrows():
                session.execute_write(create_restaurant_node, row.to_dict())
            for _, row in preferences_df.iterrows():
                session.execute_write(create_preference_node, row.to_dict())
            for _, row in users_df.iterrows():
                session.execute_write(create_user_node, row.to_dict())
            for _, row in passports_df.iterrows():
                session.execute_write(create_passport_node, row.to_dict())
            for _, row in histories_df.iterrows():
                session.execute_write(create_history_node, row.to_dict())
            
            # Additional relationships:
            # STAYED_AT (History -> Hotel) using hotels array from histories.csv
            for _, row in histories_df.iterrows():
                hist_id = row.get("history_id")
                hotels_str = row.get("hotels")
                if pd.notnull(hist_id) and isinstance(hotels_str, str) and hotels_str.strip():
                    try:
                        hotel_ids = ast.literal_eval(hotels_str)
                        for h_id in hotel_ids:
                            session.execute_write(
                                create_relationship,
                                "History", "history_id", hist_id,
                                "STAYED_AT",
                                "Hotel", "hotel_id", h_id
                            )
                    except Exception as e:
                        print("Error parsing hotels array:", e)
            
            # DINED_AT (History -> Restaurant) using restaurants array from histories.csv
            for _, row in histories_df.iterrows():
                hist_id = row.get("history_id")
                rest_str = row.get("restaurants")
                if pd.notnull(hist_id) and isinstance(rest_str, str) and rest_str.strip():
                    try:
                        rest_ids = ast.literal_eval(rest_str)
                        for r_id in rest_ids:
                            session.execute_write(
                                create_relationship,
                                "History", "history_id", hist_id,
                                "DINED_AT",
                                "Restaurant", "restaurant_id", r_id
                            )
                    except Exception as e:
                        print("Error parsing restaurants array:", e)
            
            # HAS_HOTEL_PREFERENCE (Preference -> Hotel) using top_hotels from preferences.csv
            for _, row in preferences_df.iterrows():
                pref_id = row.get("preference_id")
                top_hotels_str = row.get("top_hotels")
                if pd.notnull(pref_id) and isinstance(top_hotels_str, str) and top_hotels_str.strip():
                    try:
                        hotel_ids = ast.literal_eval(top_hotels_str)
                        for h_id in hotel_ids:
                            session.execute_write(
                                create_relationship,
                                "Preference", "preference_id", pref_id,
                                "HAS_HOTEL_PREFERENCE",
                                "Hotel", "hotel_id", h_id
                            )
                    except Exception as e:
                        print("Error parsing top_hotels array:", e)
            
            # HAS_RESTAURANT_PREFERENCE (Preference -> Restaurant) using top_restaurants
            for _, row in preferences_df.iterrows():
                pref_id = row.get("preference_id")
                top_rest_str = row.get("top_restaurants")
                if pd.notnull(pref_id) and isinstance(top_rest_str, str) and top_rest_str.strip():
                    try:
                        rest_ids = ast.literal_eval(top_rest_str)
                        for r_id in rest_ids:
                            session.execute_write(
                                create_relationship,
                                "Preference", "preference_id", pref_id,
                                "HAS_RESTAURANT_PREFERENCE",
                                "Restaurant", "restaurant_id", r_id
                            )
                    except Exception as e:
                        print("Error parsing top_restaurants array:", e)
            
            # HAS_CITY_PREFERENCE (Preference -> City) using top_cities from preferences.csv (match on City name)
            for _, row in preferences_df.iterrows():
                pref_id = row.get("preference_id")
                top_cities_str = row.get("top_cities")
                if pd.notnull(pref_id) and isinstance(top_cities_str, str) and top_cities_str.strip():
                    try:
                        city_names = ast.literal_eval(top_cities_str)
                        for cname in city_names:
                            session.execute_write(
                                create_relationship,
                                "Preference", "preference_id", pref_id,
                                "HAS_CITY_PREFERENCE",
                                "City", "City", cname
                            )
                    except Exception as e:
                        print("Error parsing top_cities array:", e)
            
            # IS_READY_TO_APPLY_VISA (Preference -> Passport) if visa_preference equals Requirement
            prefs = preferences_df.to_dict("records")
            pports = passports_df.to_dict("records")
            for pref_row in prefs:
                pref_id = pref_row["preference_id"]
                v_pref = pref_row.get("visa_preference")
                if pd.notnull(pref_id) and pd.notnull(v_pref):
                    for pport_row in pports:
                        pport_id = pport_row["passport_id"]
                        req = pport_row.get("Requirement")
                        if pd.notnull(pport_id) and pd.notnull(req):
                            if v_pref.strip() == req.strip():
                                session.execute_write(
                                    create_relationship,
                                    "Preference", "preference_id", pref_id,
                                    "IS_READY_TO_APPLY_VISA",
                                    "Passport", "passport_id", pport_id
                                )
            
            # REQUIRED_VISA_LIKE (Passport -> History) if Origin equals issued_passport
            for pport_row in pports:
                pport_id = pport_row["passport_id"]
                origin = pport_row.get("Origin")
                if pd.notnull(pport_id) and pd.notnull(origin):
                    for _, hist_row in histories_df.iterrows():
                        hist_id = hist_row["history_id"]
                        issued_p = hist_row.get("issued_passport")
                        if pd.notnull(hist_id) and pd.notnull(issued_p):
                            if origin.strip() == issued_p.strip():
                                session.execute_write(
                                    create_relationship,
                                    "Passport", "passport_id", pport_id,
                                    "REQUIRED_VISA_LIKE",
                                    "History", "history_id", hist_id
                                )
        print("Graph build complete! All relationships merged.")
    except Exception as e:
        print(f"Error building graph: {e}")
        print("Please check your CSV files and Neo4j connection")
        exit(1)

# Run the graph build once (comment out after first run if desired)
build_graph()

# ----------------------------
# 4. Representation and Retrieval (Neo4j version)
# ----------------------------
def build_representation(props, fields):
    parts = []
    for field, label in fields.items():
        value = props.get(field)
        if value is not None and str(value).strip() != "":
            parts.append(f"{label}: {value}")
    return "; ".join(parts)

def represent_city(node):
    fields = {
        "City": "City", "Country": "Country",
        "Remote connection: Average WiFi speed (Mbps per second)": "WiFi Speed",
        "Co-working spaces: Number of co-working spaces": "Co-working Spaces",
        "Accommodation: Average price of 1 bedroom apartment per month": "Apartment Price",
        "Food: Average cost of a meal at a local, mid-level restaurant": "Meal Cost",
        "Tourist attractions: Number of Things to do on Tripadvisor": "Attractions"
    }
    return build_representation(node._properties, fields)

def represent_flight(node):
    fields = {
        "Airline": "Airline", "Total Fare (EUR)": "Price",
        "Departure Airport Code": "From", "Arrival Airport Code": "To",
        "Duration (hrs)": "Duration", "Class": "Class"
    }
    return build_representation(node._properties, fields)

def represent_hotel(node):
    fields = {
        "name": "Name", "price": "Price",
        "number_reviews": "Reviews", "City": "City",
        "rating": "Rating", "address": "Address"
    }
    return build_representation(node._properties, fields)

def represent_restaurant(node):
    fields = {
        "Restaurant Name": "Name", "Cuisines": "Cuisines",
        "Average Cost for two": "Price for Two", "City": "City",
        "Aggregate rating": "Rating", "Address": "Address"
    }
    return build_representation(node._properties, fields)

def get_all_representations():
    all_nodes = []
    try:
        with driver.session() as session:
            for label, func in [
                ("City", represent_city),
                ("Flight", represent_flight),
                ("Hotel", represent_hotel),
                ("Restaurant", represent_restaurant)
            ]:
                result = session.run(f"MATCH (n:{label}) RETURN n")
                for record in result:
                    node = record["n"]
                    rep = func(node)
                    if rep:
                        all_nodes.append(rep)
    except Exception as e:
        print(f"Error getting representations: {e}")
    return list(set(all_nodes))

representations = get_all_representations()
print(f"Total representations for retrieval: {len(representations)}")

# ----------------------------
# 5. Embeddings and Retrieval
# ----------------------------
print("Computing embeddings...")
try:
    embedder = SentenceTransformer("all-MiniLM-L6-v2")
    doc_embeddings = embedder.encode(representations, convert_to_tensor=True)
    print("Embeddings computed successfully.")
except Exception as e:
    print(f"Error computing embeddings: {e}")
    exit(1)

def retrieve_documents(query, top_k=8, similarity_threshold=0.0):
    try:
        query_embedding = embedder.encode([query], convert_to_tensor=True)
        cos_scores = cosine_similarity(query_embedding.cpu().numpy(), doc_embeddings.cpu().numpy())[0]
        sorted_indices = np.argsort(cos_scores)[::-1]
        retrieved_docs = [representations[i] for i in sorted_indices[:top_k]]
        return retrieved_docs
    except Exception as e:
        print(f"Error retrieving documents: {e}")
        return []

# ----------------------------
# 6. Query Processing and Generation
# ----------------------------
print("Loading text generation model...")
try:
    generator = pipeline(
        "text-generation",
        model="gpt2",
        do_sample=True,
        temperature=0.7,
        max_new_tokens=200,
        no_repeat_ngram_size=3,
        repetition_penalty=1.2
    )
    print("Text generation model loaded successfully.")
except Exception as e:
    print(f"Error loading text generation model: {e}")
    exit(1)

def detect_query_type(query):
    query_lower = query.lower()
    if any(word in query_lower for word in ["hotel", "stay", "accommodation", "lodging"]):
        return "hotel"
    elif any(word in query_lower for word in ["restaurant", "eat", "dine", "food", "cuisine"]):
        return "restaurant"
    elif any(word in query_lower for word in ["flight", "fly", "airline", "ticket"]):
        return "flight"
    elif any(word in query_lower for word in ["city", "destination", "place", "visit", "location"]):
        return "city"
    elif any(word in query_lower for word in ["trip", "itinerary", "plan", "vacation", "holiday"]):
        return "complete_trip"
    elif any(word in query_lower for word in ["clear", "reset", "delete", "erase"]):
        return "clear_history"
    else:
        return "general"

def refine_query(raw_query):
    query_type = detect_query_type(raw_query)
    if query_type == "clear_history":
        return raw_query, query_type
        
    prompt = f"""
    Refine this travel query to be more specific for a {query_type} search:
    Original Query: {raw_query}
    Refined Query:"""
    try:
        result = generator(prompt, num_return_sequences=1)
        refined = result[0]["generated_text"].replace(prompt, "").strip().split("\n")[0].strip()
        return refined, query_type
    except Exception as e:
        print(f"Error refining query: {e}")
        return raw_query, query_type

def query_neo4j(query_type, filters=None):
    """Query Neo4j based on query type and filters"""
    if not filters:
        filters = {}
    
    results = []
    
    try:
        with driver.session() as session:
            if query_type == "hotel":
                cypher = """
                MATCH (h:Hotel)
                WHERE 1=1
                """
                if 'city' in filters:
                    cypher += f" AND h.City = '{filters['city']}'"
                if 'max_price' in filters:
                    cypher += f" AND toFloat(h.price) <= {filters['max_price']}"
                
                cypher += " RETURN h"
                result = session.run(cypher)
                for record in result:
                    results.append(record["h"])
                
            elif query_type == "restaurant":
                cypher = """
                MATCH (r:Restaurant)
                WHERE 1=1
                """
                if 'city' in filters:
                    cypher += f" AND r.City = '{filters['city']}'"
                if 'cuisine' in filters:
                    cypher += f" AND toLower(r.Cuisines) CONTAINS toLower('{filters['cuisine']}')"
                
                cypher += " RETURN r"
                result = session.run(cypher)
                for record in result:
                    results.append(record["r"])
                
            elif query_type == "flight":
                cypher = """
                MATCH (f:Flight)
                WHERE 1=1
                """
                if 'destination' in filters:
                    cypher += f" AND toLower(f.`Arrival Airport Code`) = toLower('{filters['destination']}')"
                if 'max_price' in filters:
                    cypher += f" AND toFloat(f.`Total Fare (EUR)`) <= {filters['max_price']}"
                
                cypher += " RETURN f"
                result = session.run(cypher)
                for record in result:
                    results.append(record["f"])
                
            elif query_type == "city":
                cypher = """
                MATCH (c:City)
                WHERE 1=1
                """
                if 'country' in filters:
                    cypher += f" AND toLower(c.Country) = toLower('{filters['country']}')"
                if 'wifi_speed' in filters:
                    cypher += f" AND toFloat(c.`Remote connection: Average WiFi speed (Mbps per second)`) >= {filters['wifi_speed']}"
                
                cypher += " RETURN c"
                result = session.run(cypher)
                for record in result:
                    results.append(record["c"])
    except Exception as e:
        print(f"Error querying Neo4j: {e}")
    
    return results

# ----------------------------
# 7. IMPROVED Response Generation (Neo4j version)
# ----------------------------
def generate_response(query):
    try:
        # First detect what kind of information the user wants
        refined_query, query_type = refine_query(query)
        print(f"Detected query type: {query_type}, Refined: {refined_query}")
        
        # Handle clear history command
        if query_type == "clear_history":
            global conversation_history
            conversation_history = []
            return "I've cleared our conversation history. How can I help you with your travel plans?", []
        
        # Extract filters from query
        filters = {}
        if query_type == "hotel":
            # Extract city if mentioned
            with driver.session() as session:
                result = session.run("MATCH (c:City) RETURN c.City as city")
                cities = [record["city"] for record in result]
                for city in cities:
                    if city.lower() in refined_query.lower():
                        filters['city'] = city
                        break
                        
            # Extract max price if mentioned
            if "under" in refined_query.lower() and "$" in refined_query.lower():
                try:
                    max_price = float(refined_query.split("$")[1].split()[0])
                    filters['max_price'] = max_price
                except:
                    pass
        
        elif query_type == "restaurant":
            # Extract city if mentioned
            with driver.session() as session:
                result = session.run("MATCH (c:City) RETURN c.City as city")
                cities = [record["city"] for record in result]
                for city in cities:
                    if city.lower() in refined_query.lower():
                        filters['city'] = city
                        break
                        
            # Extract cuisine type if mentioned
            cuisine_words = ["italian", "chinese", "french", "japanese", "mexican", "indian", "thai"]
            for word in cuisine_words:
                if word in refined_query.lower():
                    filters['cuisine'] = word
                    break
        
        elif query_type == "flight":
            # Extract destination if mentioned
            with driver.session() as session:
                result = session.run("MATCH (c:City) RETURN c.City as city")
                cities = [record["city"] for record in result]
                for city in cities:
                    if city.lower() in refined_query.lower():
                        filters['destination'] = city
                        break
                        
            # Extract max price if mentioned
            if "under" in refined_query.lower() and "$" in refined_query.lower():
                try:
                    max_price = float(refined_query.split("$")[1].split()[0])
                    filters['max_price'] = max_price
                except:
                    pass
        
        elif query_type == "city":
            # Extract country if mentioned
            with driver.session() as session:
                result = session.run("MATCH (c:City) RETURN c.Country as country")
                countries = [record["country"] for record in result]
                for country in countries:
                    if country.lower() in refined_query.lower():
                        filters['country'] = country
                        break
            
            # Extract WiFi speed requirement if mentioned
            if "wifi" in refined_query.lower() or "internet" in refined_query.lower():
                filters['wifi_speed'] = 50  # default high speed
        
        # Query Neo4j with filters
        neo4j_results = query_neo4j(query_type, filters)
        
        # Retrieve relevant documents based on query type
        retrieved_docs = retrieve_documents(refined_query, top_k=10)
        
        if not neo4j_results and not retrieved_docs:
            return "I couldn't find enough information about that. Could you be more specific?", []
        
        # Generate a response based on query type and Neo4j results
        if query_type == "hotel":
            if not neo4j_results:
                return "I couldn't find any hotels matching your criteria in our database. Please try a different search.", []
            
            # Sort by price
            neo4j_results.sort(key=lambda x: float(x._properties.get("price", 99999)))
            
            # Build response
            target_city = filters.get('city', 'our database')
            max_price = filters.get('max_price', 'any price')
            
            response = f"Here are the best hotel options in {target_city} under ${max_price if max_price != 'any price' else 'any price'}:\n\n"
            for i, hotel in enumerate(neo4j_results[:5]):  # Show top 5
                props = hotel._properties
                response += f"🏨 {props.get('name', 'N/A')}\n"
                response += f"   - Price: ${props.get('price', 'N/A')}\n"
                response += f"   - Reviews: {props.get('number_reviews', 'N/A')}\n"
                response += f"   - Rating: {props.get('rating', 'N/A')}\n"
                response += f"   - Address: {props.get('address', 'N/A')}\n\n"
            
            response += "Would you like:\n"
            response += "1. More details about any of these hotels\n"
            response += "2. Cheaper options in a different area\n"
            response += "3. Higher-end options with better amenities\n"
            response += "4. Something else?"
            
            return response, [represent_hotel(h) for h in neo4j_results[:5]]
            
        elif query_type == "restaurant":
            if not neo4j_results:
                cuisine = filters.get('cuisine', '')
                city = filters.get('city', 'our database')
                return f"I couldn't find any {cuisine + ' ' if cuisine else ''}restaurants matching your criteria in {city}. Please try a different search.", []
            
            # Sort by price
            neo4j_results.sort(key=lambda x: float(x._properties.get("Average Cost for two", 0))))
            
            cuisine = filters.get('cuisine', '')
            city = filters.get('city', 'various cities')
            
            response = f"Here are some excellent {cuisine if cuisine else ''} restaurant options in {city}:\n\n"
            for i, restaurant in enumerate(neo4j_results[:5]):
                props = restaurant._properties
                response += f"🍽️ {props.get('Restaurant Name', 'N/A')}\n"
                response += f"   - Cuisine: {props.get('Cuisines', 'N/A')}\n"
                response += f"   - Avg. cost for two: ${props.get('Average Cost for two', 'N/A')}\n"
                response += f"   - Rating: {props.get('Aggregate rating', 'N/A')}\n"
                response += f"   - Address: {props.get('Address', 'N/A')}\n\n"
            
            response += "Would you like to:\n"
            response += "1. Filter by a specific price range\n"
            response += "2. See options in a different area\n"
            response += "3. Get recommendations for a different cuisine\n"
            response += "4. More details about any of these"
            
            return response, [represent_restaurant(r) for r in neo4j_results[:5]]
            
        elif query_type == "flight":
            if not neo4j_results:
                return "I couldn't find any flights matching your criteria. Please try a different search.", []
            
            # Sort by price
            neo4j_results.sort(key=lambda x: float(x._properties.get("Total Fare (EUR)", 99999))))
            
            destination = filters.get('destination', 'various destinations')
            
            response = f"Here are the best flight options to {destination}:\n\n"
            for i, flight in enumerate(neo4j_results[:5]):
                props = flight._properties
                response += f"✈️ {props.get('Airline', 'N/A')}\n"
                response += f"   - From: {props.get('Departure Airport Code', 'N/A')}\n"
                response += f"   - To: {props.get('Arrival Airport Code', 'N/A')}\n"
                response += f"   - Price: ${props.get('Total Fare (EUR)', 'N/A')}\n"
                response += f"   - Duration: {props.get('Duration (hrs)', 'N/A')} hours\n"
                response += f"   - Class: {props.get('Class', 'N/A')}\n\n"
            
            response += "Would you like to:\n"
            response += "1. See flights from a specific location\n"
            response += "2. Filter by airline or flight duration\n"
            response += "3. See business class options\n"
            response += "4. Get recommendations for a different destination"
            
            return response, [represent_flight(f) for f in neo4j_results[:5]]
            
        elif query_type == "city":
            if not neo4j_results:
                return "I couldn't find any cities matching your criteria. Please try a different search.", []
            
            response = "Here are some great travel destinations:\n\n"
            for i, city in enumerate(neo4j_results[:5]):
                props = city._properties
                response += f"🌆 {props.get('City', 'N/A')}, {props.get('Country', 'N/A')}\n"
                response += f"   - Avg. apartment price: ${props.get('Accommodation: Average price of 1 bedroom apartment per month', 'N/A')}/month\n"
                response += f"   - Avg. meal cost: ${props.get('Food: Average cost of a meal at a local, mid-level restaurant', 'N/A')}\n"
                response += f"   - WiFi speed: {props.get('Remote connection: Average WiFi speed (Mbps per second)', 'N/A')} Mbps\n"
                response += f"   - Attractions: {props.get('Tourist attractions: Number of Things to do on Tripadvisor', 'N/A')} things to do\n\n"
            
            response += "Would you like more details about:\n"
            response += "1. Digital nomad-friendly cities\n"
            response += "2. Budget travel destinations\n"
            response += "3. Luxury travel options\n"
            response += "4. A specific city"
            
            return response, [represent_city(c) for c in neo4j_results[:5]]
            
        elif query_type == "complete_trip":
            # Extract destination from query
            destination = None
            duration = 5  # default
            
            # Check for duration in query
            duration_words = ["day", "week", "month"]
            for word in duration_words:
                if word in refined_query.lower():
                    try:
                        duration = int(refined_query.lower().split(word)[0].split()[-1])
                        if word == "week":
                            duration *= 7
                        elif word == "month":
                            duration *= 30
                    except:
                        pass
            
            # Find destination city
            with driver.session() as session:
                result = session.run("MATCH (c:City) RETURN c")
                for record in result:
                    city = record["c"]
                    if city._properties.get("City", "").lower() in refined_query.lower():
                        destination = city
                        break
            
            if not destination:
                return "Please specify a destination city for your trip plan (e.g., 'Plan a 5-day trip to Paris').", []
            
            # Get relevant items from Neo4j
            city_hotels = []
            city_restaurants = []
            city_flights = []
            
            with driver.session() as session:
                # Get hotels in destination city
                result = session.run(f"""
                    MATCH (h:Hotel)
                    WHERE h.City = $city
                    RETURN h
                    ORDER BY toFloat(h.price)
                    LIMIT 5
                """, city=destination._properties.get("City"))
                city_hotels = [record["h"] for record in result]
                
                # Get restaurants in destination city
                result = session.run(f"""
                    MATCH (r:Restaurant)
                    WHERE r.City = $city
                    RETURN r
                    ORDER BY toFloat(r.`Average Cost for two`)
                    LIMIT 5
                """, city=destination._properties.get("City"))
                city_restaurants = [record["r"] for record in result]
                
                # Get flights to destination city
                result = session.run(f"""
                    MATCH (f:Flight)
                    WHERE toLower(f.`Arrival Airport Code`) = toLower($city)
                    RETURN f
                    ORDER BY toFloat(f.`Total Fare (EUR)`)
                    LIMIT 1
                """, city=destination._properties.get("City"))
                city_flights = [record["f"] for record in result]
            
            # Build itinerary
            props = destination._properties
            attractions = props.get('Tourist attractions: Number of Things to do on Tripadvisor', 'many')
            
            response = f"Here's a suggested {duration}-day itinerary for {props.get('City', 'N/A')}, {props.get('Country', 'N/A')}:\n\n"
            
            # Day 1: Arrival
            response += "📅 Day 1: Arrival & First Impressions\n"
            if city_flights:
                flight_props = city_flights[0]._properties
                response += f"✈️ Flight: {flight_props.get('Airline', 'N/A')} from {flight_props.get('Departure Airport Code', 'N/A')} for ${flight_props.get('Total Fare (EUR)', 'N/A')} ({flight_props.get('Duration (hrs)', 'N/A')} hrs)\n"
            if city_hotels:
                hotel_props = city_hotels[len(city_hotels)//2]._properties
                response += f"🏨 Hotel: {hotel_props.get('name', 'N/A')} (${hotel_props.get('price', 'N/A')}/night, {hotel_props.get('rating', 'N/A')}★)\n"
                response += f"   - Address: {hotel_props.get('address', 'N/A')}\n"
            response += "   - After checking in, take a walk around the neighborhood to get oriented\n"
            if city_restaurants:
                restaurant_props = city_restaurants[0]._properties
                response += f"🍽️ Dinner: {restaurant_props.get('Restaurant Name', 'N/A')} ({restaurant_props.get('Cuisines', 'N/A')}, ${restaurant_props.get('Average Cost for two', 'N/A')} for two)\n"
                response += f"   - Rating: {restaurant_props.get('Aggregate rating', 'N/A')}★\n\n"
            
            # Day 2: Sightseeing
            response += f"📅 Day 2: Explore {props.get('City', 'N/A')}\n"
            response += "   - Morning: Visit top historical attractions (suggested: main landmarks)\n"
            response += "   - Afternoon: Take a guided walking tour or explore local markets\n"
            response += "   - Evening: Enjoy local entertainment or nightlife\n"
            if len(city_restaurants) > 1:
                restaurant_props = city_restaurants[1]._properties
                response += f"🍽️ Dinner: {restaurant_props.get('Restaurant Name', 'N/A')} ({restaurant_props.get('Cuisines', 'N/A')}, ${restaurant_props.get('Average Cost for two', 'N/A')} for two)\n\n"
            
            # Day 3: Cultural Experiences
            response += f"📅 Day 3: Cultural Immersion\n"
            response += "   - Morning: Visit museums or cultural centers\n"
            response += "   - Afternoon: Take a cooking class or craft workshop\n"
            response += "   - Evening: Attend a traditional performance\n\n"
            
            # Day 4: Day Trip
            response += f"📅 Day 4: Day Trip\n"
            response += "   - Full-day excursion to nearby attractions\n"
            response += "   - Suggested: Famous nearby sites or natural wonders\n\n"
            
            # Day 5: Relaxation & Departure
            response += f"📅 Day 5: Relaxation & Departure\n"
            response += "   - Morning: Last-minute shopping or visit favorite spots\n"
            response += "   - Afternoon: Check out from hotel\n"
            if city_flights:
                flight_props = city_flights[0]._properties
                response += f"✈️ Flight: {flight_props.get('Airline', 'N/A')} to {flight_props.get('Departure Airport Code', 'N/A')}\n\n"
            
            # Budget estimate
            total_cost = 0
            if city_flights:
                total_cost += float(city_flights[0]._properties.get("Total Fare (EUR)", 0)) * 2  # round trip
            if city_hotels:
                total_cost += float(city_hotels[len(city_hotels)//2]._properties.get("price", 0)) * duration
            if city_restaurants:
                total_cost += float(city_restaurants[0]._properties.get("Average Cost for two", 0)) * duration / 2
            # Add activities estimate
            total_cost += 50 * duration  # approx $50/day for activities
            
            response += f"💰 Estimated total cost for this trip: ${total_cost:.2f} (for one person)\n\n"
            
            response += "Would you like me to:\n"
            response += "1. Adjust this itinerary (higher/lower budget)\n"
            response += "2. Focus on specific interests (culture, food, adventure)\n"
            response += "3. Provide more detailed daily activities\n"
            response += "4. Book any of these options"
            
            retrieved = []
            if city_flights: retrieved.append(represent_flight(city_flights[0]))
            if city_hotels: retrieved.append(represent_hotel(city_hotels[len(city_hotels)//2]))
            if city_restaurants: retrieved.extend([represent_restaurant(r) for r in city_restaurants[:2]])
            retrieved.append(represent_city(destination))
            
            return response, retrieved
            
        else:
            # For general queries, use the generator with a better prompt
            prompt = f"""You are a knowledgeable travel assistant. Provide a helpful, detailed response to this travel question:

Question: {query}

Available information:
{retrieved_docs}

Response:"""
            
            result = generator(prompt, num_return_sequences=1)
            response = result[0]["generated_text"].replace(prompt, "").strip()
            
            return response, retrieved_docs
    except Exception as e:
        print(f"Error generating response: {e}")
        return "Sorry, I encountered an error while processing your request. Please try again.", []

# ----------------------------
# 8. Flask Web Interface (Enhanced version)
# ----------------------------
HTML_TEMPLATE = """
<!DOCTYPE html>
<html>
<head>
    <title>Neo4j Travel Assistant</title>
    <style>
        body {
            font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
            max-width: 1200px;
            margin: 0 auto;
            padding: 20px;
            background-color: #f5f7fa;
            color: #333;
        }
        .header {
            background-color: #4285f4;
            color: white;
            padding: 20px;
            border-radius: 8px;
            margin-bottom: 20px;
            text-align: center;
        }
        .filter-section {
            background-color: white;
            padding: 15px;
            border-radius: 8px;
            margin-bottom: 20px;
            box-shadow: 0 2px 4px rgba(0,0,0,0.1);
        }
        .filter-buttons {
            display: flex;
            gap: 10px;
            flex-wrap: wrap;
            margin-top: 10px;
        }
        .filter-button {
            padding: 8px 15px;
            background-color: #e0e0e0;
            border: none;
            border-radius: 20px;
            cursor: pointer;
            transition: background-color 0.3s;
        }
        .filter-button:hover {
            background-color: #d0d0d0;
        }
        .filter-button.active {
            background-color: #4285f4;
            color: white;
        }
        .chat-container {
            background-color: white;
            border-radius: 8px;
            padding: 20px;
            margin-bottom: 20px;
            box-shadow: 0 2px 4px rgba(0,0,0,0.1);
            height: 500px;
            overflow-y: auto;
            white-space: pre-wrap;
        }
        .message {
            margin-bottom: 15px;
            padding: 10px 15px;
            border-radius: 18px;
            max-width: 80%;
            word-wrap: break-word;
        }
        .user-message {
            background-color: #e3f2fd;
            margin-left: auto;
            border-bottom-right-radius: 4px;
        }
        .bot-message {
            background-color: #f1f1f1;
            margin-right: auto;
            border-bottom-left-radius: 4px;
            white-space: pre-wrap;
        }
        .data-section {
            background-color: white;
            border-radius: 8px;
            padding: 20px;
            margin-bottom: 20px;
            box-shadow: 0 2px 4px rgba(0,0,0,0.1);
            max-height: 300px;
            overflow-y: auto;
        }
        .data-item {
            padding: 10px;
            border-bottom: 1px solid #eee;
            font-family: monospace;
        }
        .input-section {
            display: flex;
            gap: 10px;
        }
        #user-input {
            flex-grow: 1;
            padding: 12px;
            border: 1px solid #ddd;
            border-radius: 8px;
            font-size: 16px;
        }
        #submit-button {
            padding: 12px 20px;
            background-color: #4285f4;
            color: white;
            border: none;
            border-radius: 8px;
            cursor: pointer;
            font-size: 16px;
        }
        #submit-button:hover {
            background-color: #3367d6;
        }
        .query-type-indicator {
            font-size: 14px;
            color: #666;
            margin-top: 5px;
            font-style: italic;
        }
        .clear-button {
            padding: 8px 15px;
            background-color: #f44336;
            color: white;
            border: none;
            border-radius: 8px;
            cursor: pointer;
            font-size: 14px;
            margin-top: 10px;
        }
        .clear-button:hover {
            background-color: #d32f2f;
        }
        .neo4j-info {
            background-color: #008cc1;
            color: white;
            padding: 10px;
            border-radius: 5px;
            margin-top: 10px;
            font-size: 14px;
        }
        .error-message {
            color: #d32f2f;
            background-color: #ffebee;
            padding: 10px;
            border-radius: 5px;
            margin: 10px 0;
        }
    </style>
</head>
<body>
    <div class="header">
        <h1>Neo4j Travel Assistant</h1>
        <p>Get personalized travel recommendations powered by Neo4j graph database</p>
        <div class="neo4j-info">
            Connected to Neo4j at {{ NEO4J_URI }} with {{ node_count }} nodes in database
        </div>
    </div>
    
    <div class="filter-section">
        <h3>Not sure what to ask? Try these Neo4j-powered queries:</h3>
        <div class="filter-buttons">
            <button class="filter-button" onclick="setQuery('Best hotels in New York under $200')">Hotels</button>
            <button class="filter-button" onclick="setQuery('Italian restaurants in London')">Restaurants</button>
            <button class="filter-button" onclick="setQuery('Cheapest flights to Dubai next month')">Flights</button>
            <button class="filter-button" onclick="setQuery('Best digital nomad cities with good WiFi')">Destinations</button>
            <button class="filter-button" onclick="setQuery('Plan a complete 5-day trip to Istanbul')">Complete Trip</button>
        </div>
        <button class="clear-button" onclick="setQuery('clear history')">Clear Conversation</button>
    </div>
    
    <div class="chat-container" id="chat-container">
        {% for msg in history %}
            <div class="message {% if msg.sender == 'User' %}user-message{% else %}bot-message{% endif %}">
                <strong>{{ msg.sender }}:</strong> {{ msg.text }}
                {% if msg.query_type %}
                <div class="query-type-indicator">Detected as: {{ msg.query_type }}</div>
                {% endif %}
            </div>
        {% endfor %}
    </div>
    
    <div class="data-section">
        <h3>Neo4j Data Details</h3>
        {% if retrieved_data %}
            {% for doc in retrieved_data %}
                <div class="data-item">{{ doc }}</div>
            {% endfor %}
        {% else %}
            <div class="data-item">No data retrieved yet. Ask about hotels, restaurants, flights or destinations.</div>
        {% endif %}
    </div>
    
    <form method="post" class="input-section">
        <input type="text" id="user-input" name="question" placeholder="Ask about hotels, flights, restaurants or destinations..." required>
        <input type="submit" id="submit-button" value="Send">
    </form>
    
    <script>
        function setQuery(query) {
            document.getElementById('user-input').value = query;
            if (query.toLowerCase().includes('clear')) {
                document.forms[0].submit();
            }
            document.getElementById('user-input').focus();
        }
        
        // Auto-scroll chat to bottom
        var chatContainer = document.getElementById("chat-container");
        chatContainer.scrollTop = chatContainer.scrollHeight;
    </script>
</body>
</html>
"""

def get_node_count():
    try:
        with driver.session() as session:
            result = session.run("MATCH (n) RETURN COUNT(n) AS count")
            return result.single()["count"]
    except Exception as e:
        print(f"Error getting node count: {e}")
        return 0

@app.route("/", methods=["GET", "POST"])
def index():
    global conversation_history
    retrieved_data = []
    node_count = get_node_count()
    
    if request.method == "POST":
        question = request.form["question"]
        query_type = detect_query_type(question)
        conversation_history.append({
            "sender": "User", 
            "text": question,
            "query_type": query_type.replace("_", " ").title()
        })
        
        answer, retrieved_data = generate_response(question)
        conversation_history.append({
            "sender": "Assistant", 
            "text": answer
        })
    
    return render_template_string(
        HTML_TEMPLATE, 
        history=conversation_history, 
        retrieved_data=retrieved_data,
        NEO4J_URI=NEO4J_URI,
        node_count=node_count
    )

# ----------------------------
# 9. Run the Application
# ----------------------------
if __name__ == "__main__":
    print("\nStarting Flask application...")
    print(f"Access the web interface at http://localhost:5001")
    print("Press CTRL+C to stop the server\n")
    app.run(host="0.0.0.0", port=5001, debug=False)

SyntaxError: unmatched ')' (2826302903.py, line 637)